## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/notebook?scriptVersionId=346297722
- v4_ood
- base_model
- 8000token

In [5]:
import polars as pl
from pathlib import Path
import sys
# sys.path.append(str('d:/qwen_reasoning_test/ArrayTask'))
sys.path.append(str(Path.cwd().parent.parent))
from src.gen_task import RAW_RULES, RAW_OOD_RULES_V1

RULE_CONFIG = RAW_RULES
# RULE_CONFIG = RAW_OOD_RULES_V1

RULES = [
    {
        "id": f"{i:03d}",
        "primitives": rule,
    }
    for i, rule in enumerate(RULE_CONFIG)
]

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[6, 0, 4, 8, 7, 6, 4, 7]

Output:
[7, 4, 6, 7, 8, 4, 0, 6]

Example 2

Input:
[9, 3, 8, 2, 4, 2, 1]

Output:
[1, 2, 4, 2, 8, 3, 9]

Example 3

Input:
[4, 8, 9, 2, 4, 1, 1, 5, 7]

Output:
[7, 5, 1, 1, 4, 2, 9, 8, 4]

Example 4

Input:
[1, 5, 6, 5, 9, 3, 8, 7, 7]

Output:
[7, 7, 8, 3, 9, 5, 6, 5, 1]

Example 5

Input:
[4, 0, 8, 0, 1, 6, 0, 9, 7]

Output:
[7, 9, 0, 6, 1, 0, 8, 0, 4]

Query

Input:
[3, 5, 1, 3, 9, 3, 3]

Output:
Raw_output: 
<think>
Example 1:
Input: [6, 0, 4, 8, 7, 6, 4, 7]
Step 1 (reverse): [7, 4, 6, 7, 8, 4, 0, 6]
Output: [7, 4, 6, 7, 8, 4, 0, 6]

Example 2:
Input: [9, 3, 8, 2, 4, 2, 1]
Step 1 (reverse): [1, 2, 4, 2, 8, 3, 9]
Output: [1, 2, 4, 2, 8, 3, 9]

Example 3:
Input: [4, 8, 9, 2, 4, 1, 1, 5, 7]


In [6]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
print(f"acc: {match_count / len(df)}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 720
match_count: 511
acc: 0.7097222222222223


# 正解

In [7]:
match_results = []

for rule in RULES:
    if rule["id"] in [i for i, j in match_counts.items()]:
        for i, j in match_counts.items():
            if rule["id"] == i:
                match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})
    else:
        match_results.append({"task_id": rule["id"], "count": 0, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["task_id"], reverse=True)

records = []
for x in match_results:
    records.append({"task_id": x["task_id"], "rule": x["rule"], "count": x["count"]})
    print(x)

heatmap_df = pl.DataFrame(records)
pl.Config.set_tbl_rows(-1)

heatmap_df


{'task_id': '071', 'count': 4, 'rule': ['shift_left', 'take_odd_positions']}
{'task_id': '070', 'count': 7, 'rule': ['shift_left', 'take_even_positions']}
{'task_id': '069', 'count': 5, 'rule': ['shift_left', 'mirror']}
{'task_id': '068', 'count': 3, 'rule': ['shift_left', 'adjacent_sum']}
{'task_id': '067', 'count': 1, 'rule': ['shift_left', 'subtract_next']}
{'task_id': '066', 'count': 9, 'rule': ['shift_left', 'mod_3']}
{'task_id': '065', 'count': 7, 'rule': ['shift_left', 'mod_2']}
{'task_id': '064', 'count': 4, 'rule': ['shift_left', 'add_3']}
{'task_id': '063', 'count': 5, 'rule': ['shift_left', 'add_2']}
{'task_id': '062', 'count': 0, 'rule': ['shift_left', 'add_1']}
{'task_id': '061', 'count': 1, 'rule': ['shift_left', 'multiply_3']}
{'task_id': '060', 'count': 10, 'rule': ['shift_left', 'multiply_2']}
{'task_id': '059', 'count': 2, 'rule': ['shift_left', 'pop_left']}
{'task_id': '058', 'count': 10, 'rule': ['shift_left', 'pop_right']}
{'task_id': '057', 'count': 10, 'rule': ['

task_id,rule,count
str,list[str],i64
"""071""","[""shift_left"", ""take_odd_positions""]",4
"""070""","[""shift_left"", ""take_even_positions""]",7
"""069""","[""shift_left"", ""mirror""]",5
"""068""","[""shift_left"", ""adjacent_sum""]",3
"""067""","[""shift_left"", ""subtract_next""]",1
"""066""","[""shift_left"", ""mod_3""]",9
"""065""","[""shift_left"", ""mod_2""]",7
"""064""","[""shift_left"", ""add_3""]",4
"""063""","[""shift_left"", ""add_2""]",5


# 不正解

In [8]:
# results = []
# for i, j in counts.items():
#     for rule in RULES:
#         if rule["id"] == i:
#             # print(i, j, rule)
#             results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

# results = sorted(results, key=lambda x: x["task_id"], reverse=True)
# results